# Lectura de los datos

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/digit/train.csv').head(1000)

In [2]:
print("El conjunto de datos tiene {} filas y {} columnas".format(df.shape[0],df.shape[1]))
df.head()

El conjunto de datos tiene 1000 filas y 785 columnas


,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Graficando el __primer__ número:
<img src="./img/numero.png"/>

In [ ]:
y = df.label.values
x = df.drop("label",axis=1)

In [ ]:
print("El conjunto de datos tiene {} filas y {} columnas".format(x.shape[0],x.shape[1]))
x.head(2)

In [ ]:
i = 1
plt.matshow(x.iloc[i].values.reshape(28,28))
print("El numero escrito es {}".format(y[i]))

# La red neuronal

## Preparando los datos de entrada y salida

Cuando hablamos de números, solemos verlos como ordinales
$$\overrightarrow{0, 1, 2, 3, 4, 5, 6, 7, 8, 9 }$$



$$0  \rightarrow \left  [1, 0, 0, 0, 0, 0, 0, 0, 0, 0 \right ]$$
$$3  \rightarrow \left  [0, 0, 0, 1, 0, 0, 0, 0, 0, 0 \right ]$$
$$4  \rightarrow \left  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0 \right ]$$
$$9  \rightarrow \left  [0, 0, 0, 0, 0, 0, 0, 0, 0, 1 \right ]$$

In [ ]:
def one_hot(j):
    e = np.zeros(10)
    e[j] = 1.0
    return e

#aplicamos la función one_hot
y_n = np.array([one_hot(number) for number in y])

y_n

In [ ]:
#Solo para comparar el vector nuevo contra el numero original
for a,b in zip(y_n[:5],y[:5]): print(b," se convierte en ",a)

Es muy importante normalizar los datos de entrada

In [ ]:
#Normalizamos x
x = x/255

# Construyendo la red neuronal

In [ ]:
lr = 0.01
w0 = np.random.randn(x.shape[1],16)
w1 = np.random.randn(16, 16)
w2 = np.random.randn(16, 10)
b0 = np.random.randn(1, 16)
b1 = np.random.randn(1, 16)
b2 = np.random.randn(1, 10)
Z0,z1,z2,a0,a1,a2,a3 = [],[],[],[],[],[],[]
y = y
mse=0
output = np.zeros(y.shape)
errors = []

In [ ]:
w0.shape

In [ ]:
b0.shape

In [ ]:
w1.shape

In [ ]:
b1.shape

In [ ]:
w2.shape

In [ ]:
b2.shape

In [ ]:
def sigmoid(t):
    return 1/(1+np.exp(-t))

def sigmoid_derivative(p):
    return sigmoid(p) * sigmoid(1 - p)

In [ ]:
def feedforward(x_input):
    #np.dot regresa el producto punto de dos arreglos.
    global a0,z0,a1,z1,a2,z2,a3
    a0 = x_input
    z0 = np.dot(a0,w0) + b0
    a1 = sigmoid(z0)
    z1 = np.dot(a1,w1) + b1
    a2 = sigmoid(z1)
    z2 = np.dot(a2,w2) + b2
    a3 = sigmoid(z2)
    output = a3
    return output

In [ ]:
def backprop():
    # Aplicando la regla de la cadena para la función de perdida respecto a los pesos 2 y 1.
    # .T devuelve la matriz traspuesta
    global mse,w0,w1,w2,b0,b1,b2
    mse = np.sum((y_n - output)**2)
    errors.append(mse)

    delta2 = -(y_n - output) * sigmoid_derivative(z2)
    d_w2 = np.dot(a2.T,delta2)
    d_b2 = delta2
    
    delta1 = np.dot(delta2,w2.T) * sigmoid_derivative(z1)
    d_w1 = np.dot(a1.T,delta1)
    d_b1 = delta1
    
    delta0 = np.dot(delta1,w1.T) * sigmoid_derivative(z0)
    d_w0 = np.dot(a0.T,delta0)
    d_b0 = delta0

    w2 = w2 - lr * d_w2
    w1 = w1 - lr * d_w1
    w0 = w0 - lr * d_w0

    b2 = b2 - lr * d_b2.mean(axis=0).reshape(b2.shape)
    b1 = b1 - lr * d_b1.mean(axis=0).reshape(b1.shape)
    b0 = b0 - lr * d_b0.mean(axis=0).reshape(b0.shape)

In [ ]:
for i in range(10000):
    output = feedforward(x)
    backprop()
    if i % 100 == 0:
        print("Epoch: {}, mse: {}".format(i,mse))

In [ ]:
output[0]

In [ ]:
y_hat = output.argmax(axis=1)
y_hat[0]

In [ ]:
y[:5]

In [ ]:
y != y_hat

In [ ]:
(y != y_hat).sum()

De los 10,000 valores solamente existen 619 que no pudo predecir correctamente.

In [ ]:
errores = x[y != y_hat].index
x[y != y_hat]

In [ ]:
for i in errores:
    plt.matshow(x.iloc[i].values.reshape(28,28))
    print("El numero escrito es {}".format(y[i]))
    print("El numero predicho es {}".format(y_hat[i]))
    plt.show()